# Notebook 1 – Multi 75-G Quick Start

This notebook is the entry point to `eb_cr_afm` and demonstrates the complete workflow
for simulating a **Budget Sensors Multi 75-G** cantilever probe (k = 2.8 N/m, f_free ≈ 75 kHz).

## What you will learn

1. How to instantiate cantilever, contact, excitation, and solver parameters using the built-in `default_params_multi75()` helper
2. How to run a full frequency sweep in **free air** and **in contact** with a stiff sample
3. How to extract the contact-resonance frequency, quality factor Q, and amplitude by fitting a simple harmonic oscillator (SHO) model to each peak
4. How to compute the **contact-resonance shape factor** κ, which quantifies how efficiently a given dynamic mode couples to the contact stiffness

## Background

The Euler–Bernoulli beam model treats the cantilever as a 1-D elastic beam clamped at the base and free at the tip. When the tip makes contact with a sample, the tip boundary conditions change: vertical and lateral springs (k₁, k₂) are added at the free end, shifting the resonance frequencies upward. By tracking the contact-resonance frequency as a function of drive, one can infer the local sample stiffness.

All plots in this notebook are **interactive Plotly figures** — hover over any point to read exact values, click legend entries to toggle traces, and use the toolbar to pan or zoom.

All figures are also saved to `../results/` as static PNG images and interactive HTML files.
This notebook is compatible with the Bradler-style axial tip offset parameter `cant.tip_offset`; when that offset is nonzero, the contact location is `cant.contact_x = L - tip_offset`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../src'))

import numpy as np
from eb_cr_afm import (
    default_params_multi75,
    ContactParams, SolverOptions,
    frequency_sweep, sho_analysis,
    compute_shape_factor,
    plot_transfer_function_plotly,
)

# Create results directory (relative to the examples/ folder)
RESULTS_DIR = os.path.join(os.getcwd(), '../results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('eb_cr_afm loaded OK')
print(f'Results will be saved to: {os.path.abspath(RESULTS_DIR)}')

## 1 – Cantilever parameters

`default_params_multi75()` returns four pre-configured dataclass objects:

| Object | Type | Description |
|--------|------|-------------|
| `cant` | `CantileverParams` | Geometry (L, w, H), mechanical properties (k, m, EI), tilt angle φ |
| `contact` | `ContactParams` | Contact stiffnesses k₁, k₂ and damping c₁, c₂ |
| `exc` | `ExcitationParams` | Drive voltage V_AC, sample potential ΔV, tip force f₀ |
| `opts` | `SolverOptions` | Frequency range, grid density, air damping g, solver backend |

The cantilever mass `m` is back-calculated from the geometry so that the first
free Euler–Bernoulli resonance frequency exactly matches **75 kHz**.

Key Multi 75-G properties:
- Length L = 225 µm, width w = 28 µm, tip height H = 14 µm
- Static spring constant k = 2.8 N/m
- Tilt angle φ = 11° (standard AFM head geometry)

In [ ]:
cant, contact, exc, opts = default_params_multi75()

print(f'L   = {cant.L*1e6:.0f} µm')
print(f'k   = {cant.k:.1f} N/m')
print(f'm   = {cant.m*1e12:.1f} ng')
print(f'phi = {np.degrees(cant.phi):.1f}°')
print(f'EI  = {cant.EI:.3e} N·m²')
print()

# Verify free resonance from beam formula
beta1 = 1.875 / cant.contact_x
rhoA  = cant.m / cant.contact_x
f_free_check = beta1**2 * np.sqrt(cant.EI / rhoA) / (2 * np.pi)
print(f'First free resonance (calc) = {f_free_check/1e3:.1f} kHz  ✓')

## 2 – Frequency sweep: free cantilever

To simulate the **free cantilever** (no tip–sample contact) we set the contact
stiffness k₁ to a very small value (0.01 N/m ≈ 0). This preserves the
boundary-condition structure of the solver without introducing a true singularity.

The sweep runs `frequency_sweep()` over the range defined in `opts`
(50–400 kHz for Multi 75-G); here we refine the step to 100 Hz so that the sharp
peaks are resolved well enough for a reliable Q fit. At each frequency the beam equation is solved (analytical solver by default)
and the tip slope amplitude |z'(L1)| and phase are recorded.

**Expected result:** A single sharp peak at ≈ 75 kHz, confirming the free-cantilever
resonance matches the nominal specification. The quality factor (a few hundred)
is set by the air damping coefficient `g` in `SolverOptions` (2000 s⁻¹ in this preset).

In [ ]:
import dataclasses

# Use a 100 Hz frequency step so the narrow resonance peaks are well resolved.
opts = dataclasses.replace(opts, f_step=100.0)

contact_free = ContactParams(k1=0.01, k2=0.01, c1=0.0, c2=0.0, psi=0.0)
# The free peak is only ~300 Hz wide, so sweep a narrow window with a 25 Hz step.
opts_free = dataclasses.replace(opts, f_start=60e3, f_stop=90e3, f_step=25.0)
sweep_free = frequency_sweep(cant, contact_free, exc, opts_free)

# sho_analysis returns every peak it finds; take the strongest one.
sho_free = max(sho_analysis(sweep_free, quantity="slope"), key=lambda m: m["A_res"])
print("Free cantilever:")
print(f"  f_res = {sho_free['f_res']/1e3:.2f} kHz")
print(f"  |z'|  = {sho_free['A_res']*1e6:.3g} µrad at the peak")
print(f"  Q     = {sho_free['Q']:.0f}")


## 3 – Frequency sweep: in contact

Now we place the tip in contact with a stiff surface by setting k₁ = 280 N/m
(100 × the cantilever spring constant). This is representative of a moderately
stiff polymer or thin oxide film.

The lateral stiffness is set to k₂ = 0.85 × k₁ — the "isotropic surface" limit
where the Poisson ratio effect couples vertical and lateral contact stiffnesses.

**Expected result:** The contact resonance shifts to ≈ 300 kHz (about 4× the free
frequency) and the Q factor increases significantly because the contact stiffening
narrows the resonance peak relative to the fixed air-damping background.

In [ ]:
contact_stiff = ContactParams(k1=280.0, k2=0.85*280.0, c1=0.0, c2=0.0, psi=0.0)
sweep_contact = frequency_sweep(cant, contact_stiff, exc, opts)

sho_contact = max(sho_analysis(sweep_contact, quantity="slope"), key=lambda m: m["A_res"])
# Refine: re-sweep ±5 kHz around the peak with a 10 Hz step for an accurate Q.
opts_zoom = dataclasses.replace(opts, f_start=sho_contact["f_res"] - 5e3,
                                f_stop=sho_contact["f_res"] + 5e3, f_step=10.0)
sweep_zoom = frequency_sweep(cant, contact_stiff, exc, opts_zoom)
sho_contact = max(sho_analysis(sweep_zoom, quantity="slope"), key=lambda m: m["A_res"])
print("In contact (k1 = 280 N/m):")
print(f"  f_res = {sho_contact['f_res']/1e3:.2f} kHz")
print(f"  |z'|  = {sho_contact['A_res']*1e6:.3g} µrad at the peak (d1 = 1 pm)")
print(f"  Q     = {sho_contact['Q']:.0f}")


## 4 – Transfer function: tip slope z'(L1)

The **tip slope** z'(L1) = dz/dx evaluated at the cantilever contact point `L1 = cant.contact_x` is
the physical observable detected by the optical lever (beam deflection) in a
standard AFM setup. The photodetector signal is proportional to the angle of the
reflected laser beam, which equals the local slope of the cantilever surface.

Both free (blue) and contact (red) sweeps are overlaid on a single figure.
The amplitude is plotted on a **log scale** to allow both the free (small) and
contact (large) peaks to be seen simultaneously. The phase displays the 180° phase step
at each resonance frequency characteristic of a harmonic oscillator.

**Tip:** Click legend entries to hide/show individual traces. Hover for exact values.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Tip slope amplitude", "Phase"))

for sweep, name, color in [
    (sweep_free,    "Free",    "#1f77b4"),
    (sweep_contact, "Contact", "#d62728"),
]:
    f_kHz = sweep['freqs'] / 1e3
    fig.add_trace(go.Scatter(x=f_kHz, y=sweep['amplitudes_slope'] * 1e6,
                             name=name, line=dict(color=color, width=2),
                             showlegend=True), row=1, col=1)
    fig.add_trace(go.Scatter(x=f_kHz, y=sweep['phases_slope'],
                             name=name, line=dict(color=color, width=2, dash='dash'),
                             showlegend=False), row=2, col=1)

fig.update_xaxes(title_text="Frequency (kHz)", row=2, col=1)
fig.update_yaxes(title_text="|z'(L1)|  (µrad)  — tip slope", type="log", row=1, col=1)
fig.update_yaxes(title_text="Phase (°)", row=2, col=1)
fig.update_layout(height=500, hovermode='x unified', template='plotly_white',
                  title='Multi 75-G: Free vs In-Contact — Tip Slope z\'(L)')
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb01_tip_slope_transfer_function.html'))
print('Saved: results/nb01_tip_slope_transfer_function.html')

## 4b – Transfer function: tip displacement z(L1)

While z'(L1) is the measurable signal in most AFM setups, the **tip displacement**
z(L1) gives a more intuitive picture of the physical motion amplitude.

Both quantities are normalised to a unit drive amplitude d₁ = 1 pm, so:
- **Slope plot**: rad/pm — how many radians of tip angle per pm of sample drive
- **Displacement plot**: pm/pm = dimensionless (shown as |z(L1)| in picometres per pm drive)

The displacement plot is particularly useful for comparing the absolute magnitude of
cantilever motion before and after contact engagement. Note that contact dramatically
amplifies the tip response at the resonance frequency.

In [ ]:
import numpy as np

# Extract tip displacement |z(L1)| from stored per-frequency results (no re-solve needed)
z_tip_free    = np.array([np.abs(r['z'][-1]) for r in sweep_free['results']])
z_tip_contact = np.array([np.abs(r['z'][-1]) for r in sweep_contact['results']])

fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=("Tip displacement amplitude", "Phase"))

for f_kHz, z_tip, phases, name, color in [
    (sweep_free['freqs']    / 1e3, z_tip_free,    sweep_free['phases_disp'],    "Free",    "#1f77b4"),
    (sweep_contact['freqs'] / 1e3, z_tip_contact, sweep_contact['phases_disp'], "Contact", "#d62728"),
]:
    fig2.add_trace(go.Scatter(x=f_kHz, y=z_tip / 1e-12,
                              name=name, line=dict(color=color, width=2),
                              showlegend=True), row=1, col=1)
    fig2.add_trace(go.Scatter(x=f_kHz, y=phases,
                              name=name, line=dict(color=color, width=2, dash='dash'),
                              showlegend=False), row=2, col=1)

fig2.update_xaxes(title_text="Frequency (kHz)", row=2, col=1)
fig2.update_yaxes(title_text="|z(L1)| (pm)  — tip displacement", type="log", row=1, col=1)
fig2.update_yaxes(title_text="Phase (°)", row=2, col=1)
fig2.update_layout(height=500, hovermode='x unified', template='plotly_white',
                   title='Multi 75-G: Free vs In-Contact — Tip Displacement z(L1)')
fig2.show()

# Save figure
fig2.write_html(os.path.join(RESULTS_DIR,  'nb01_tip_displacement_transfer_function.html'))
print('Saved: results/nb01_tip_displacement_transfer_function.html')

## 5 – Shape factor at the contact resonance

The **shape factor** κ is defined as the ratio of the dynamic effective stiffness
to the static effective stiffness at the tip location:

$$\kappa = \frac{S_\text{dynamic}}{S_\text{static}} = \frac{|z'(L1)|_\text{CR}}{|z'(L1)|_\text{static}}$$

Physically, κ > 1 means the dynamic mode shape amplifies the coupling to contact
stiffness compared to a purely static indentation, increasing sensitivity.
κ < 1 means reduced coupling — less efficient for stiffness imaging.

The shape factor depends on the mode shape at the contact resonance frequency,
which in turn depends on k₁, φ, and the cantilever geometry. For the first
contact resonance of the Multi 75-G with k₁ = 280 N/m, we expect κ < 1 because
the mode shape has significant curvature away from the tip.

In [ ]:
from eb_cr_afm import solve_single_frequency

# Re-solve at the contact resonance frequency with a fine spatial grid
# to get an accurate mode shape for the shape factor calculation
f_contact_res = sho_contact['f_res']
res_at_peak = solve_single_frequency(f_contact_res, cant, contact_stiff, exc, opts)

kappa = compute_shape_factor(res_at_peak, cant)
print(f'Contact resonance: f = {f_contact_res/1e3:.2f} kHz')
print(f'Shape factor κ    = {kappa:.3f}')
print(f'(κ > 1 → enhanced sensitivity vs static)')
print()
print('Summary table:')
print(f'  Free resonance:    {sho_free["f_res"]/1e3:.2f} kHz   Q = {sho_free["Q"]:.0f}')
print(f'  Contact resonance: {sho_contact["f_res"]/1e3:.2f} kHz   Q = {sho_contact["Q"]:.0f}')
print(f'  Frequency shift:   {(sho_contact["f_res"]-sho_free["f_res"])/1e3:.2f} kHz  '
      f'({sho_contact["f_res"]/sho_free["f_res"]:.2f}×)')
print(f'  Shape factor κ:    {kappa:.3f}')